# 05 — Custom Rules via the YAML DSL

COGANT ships with ~18 built-in translation rules that map `ProgramGraph` nodes to Active Inference roles. For domain-specific vocabularies you rarely need to write Python — the **YAML rule DSL** lets you declare new rules as data.

This notebook walks through:
1. Writing a YAML rule string in-line.
2. Parsing it with `load_rules_from_dict` and compiling it with `compile_ruleset`.
3. Applying the compiled rules to a small synthetic `ProgramGraph`.
4. Inspecting which nodes matched and at what confidence.

Run from the `cogant/` directory:
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/05_custom_rules.ipynb
```

## 1. A YAML rule string

Each rule has a `name`, `role`, `confidence`, and a list of `conditions`. Every condition must match for the rule to fire; the supported condition keys are `node_kind`, `name_pattern` (glob), `has_method`, and `edge_type`.

Below we write three rules that together implement a tiny "button actuator" ontology.

In [ ]:
from __future__ import annotations

RULES_YAML = """
rules:
  - name: ButtonClassIsAction
    role: ACTION
    confidence: 0.9
    description: Any class whose name ends in Button is classified as an ACTION.
    conditions:
      - node_kind: class
        name_pattern: '*Button'

  - name: SensorClassIsObservation
    role: OBSERVATION
    confidence: 0.85
    description: Any class whose name starts with Sensor is classified as an OBSERVATION.
    conditions:
      - node_kind: class
        name_pattern: 'Sensor*'

  - name: TickMethodIsPolicy
    role: POLICY
    confidence: 0.75
    description: A method literally named `tick` is a POLICY (decision loop).
    conditions:
      - node_kind: method
        name_pattern: 'tick'
"""
print(RULES_YAML)

## 2. Load and compile the rule set

`load_rules_from_dict` accepts an already-parsed Python dict; we parse the YAML via `yaml.safe_load` first. The compiler turns each declarative rule into a `CompiledRule` object with a `.match(node, graph)` method.

In [ ]:
import yaml

from cogant.translate.dsl import load_rules_from_dict, compile_ruleset

yaml_data = yaml.safe_load(RULES_YAML)
ruleset = load_rules_from_dict(yaml_data)
print(f"Loaded {len(ruleset.rules)} rule(s):")
for rule in ruleset.rules:
    print(f"  - {rule.name:<25}  role={rule.role:<12} conf={rule.confidence}")

compiled = compile_ruleset(ruleset)
print(f"\nCompiled {len(compiled)} CompiledRule object(s).")

## 3. Build a synthetic `ProgramGraph`

We construct a tiny graph by hand so we can reason about every match without scanning a real repository. The graph contains:

- `StartButton` (class) — should match `ButtonClassIsAction`.
- `SensorTemperature` (class) — should match `SensorClassIsObservation`.
- `Controller.tick` (method on a plain `Controller` class) — should match `TickMethodIsPolicy`.
- `Controller.update` (method) — should match nothing.

In [ ]:
from cogant.schemas.core import Edge, EdgeKind, Node, NodeKind
from cogant.schemas.graph import GraphMetadata, ProgramGraph

graph = ProgramGraph(metadata=GraphMetadata(repo_uri="synthetic://tutorial"))

nodes_spec = [
    ("n_sb",  NodeKind.CLASS,    "StartButton",       "demo.StartButton"),
    ("n_st",  NodeKind.CLASS,    "SensorTemperature", "demo.SensorTemperature"),
    ("n_ctl", NodeKind.CLASS,    "Controller",        "demo.Controller"),
    ("n_tick", NodeKind.METHOD,  "tick",              "demo.Controller.tick"),
    ("n_upd",  NodeKind.METHOD,  "update",            "demo.Controller.update"),
]
for nid, kind, name, qname in nodes_spec:
    graph.add_node(Node(id=nid, kind=kind, name=name, qualified_name=qname))

# Wire the methods under the Controller class via CONTAINS edges.
graph.add_edge(Edge(id="e1", source_id="n_ctl", target_id="n_tick", kind=EdgeKind.CONTAINS))
graph.add_edge(Edge(id="e2", source_id="n_ctl", target_id="n_upd",  kind=EdgeKind.CONTAINS))

print(f"Graph has {len(graph.nodes)} nodes and {len(graph.edges)} edges.")

## 4. Apply the compiled rules

For each node we score it against every compiled rule. Non-zero scores are matches; we record them for reporting.

In [ ]:
matches: list[dict] = []
for node in graph.nodes.values():
    for rule in compiled:
        score = rule.match(node, graph)
        if score > 0.0:
            matches.append({
                "node": node.name,
                "kind": node.kind.value,
                "rule": rule.name,
                "role": rule.role,
                "confidence": score,
            })

print(f"Total matches: {len(matches)}")
for m in matches:
    print(
        f"  {m['node']:<20} ({m['kind']:<7}) → {m['role']:<12} "
        f"by {m['rule']:<25} conf={m['confidence']:.2f}"
    )

### Sanity-check the matches

Three nodes should have matched (`StartButton`, `SensorTemperature`, `Controller.tick`); the two others (`Controller`, `Controller.update`) should have no match.

In [ ]:
matched_node_names = {m["node"] for m in matches}
assert matched_node_names == {"StartButton", "SensorTemperature", "tick"}, (
    f"Unexpected match set: {matched_node_names}"
)
print("OK — matches are exactly the nodes we expected.")

## 5. Where to go from here

- Save the YAML to a file and load with `load_rules_from_yaml(path)`.
- Combine DSL rules with the built-in `TranslationEngine` by registering a rule plugin (see notebook 06).
- Use `edge_type` conditions to capture dataflow patterns, e.g. a rule that fires when a method has an outgoing `writes` edge.
- Keep rule confidences under 1.0 so higher-priority built-ins can still override them when appropriate.